# 15. Feature Profiling — Target Separation, WoE & Information Value

Feature profiling evaluates the discriminative power of individual attributes with respect to a supervised target variable. In credit risk, marketing analytics, and fraud detection, **Weight of Evidence (WoE)** and **Information Value (IV)** are the industry benchmarks.

---

### Weight of Evidence (WoE) & Information Value (IV)
For each bin $i$ of a feature:
$$\text{WoE}_i = \ln\left(\frac{\% \text{ Good}_i}{\% \text{ Bad}_i}\right)$$
$$\text{IV} = \sum_{i=1}^k \left(\% \text{ Good}_i - \% \text{ Bad}_i\right) \times \text{WoE}_i$$

| Information Value (IV) | Predictive Power |
| :--- | :--- |
| $< 0.02$ | Unpredictive / Useless |
| $0.02 - 0.10$ | Weak Predictor |
| $0.10 - 0.30$ | Medium Predictor |
| $0.30 - 0.50$ | Strong Predictor |
| $> 0.50$ | Suspicious / Too Good (Check for Data Leakage) |

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 2000

# Continuous feature: credit utilization percentage
utilization = np.random.uniform(0.05, 0.95, size=n)

# Binary target: 1 = Default (Bad), 0 = Non-Default (Good)
default_prob = 1.0 / (1.0 + np.exp(-(utilization - 0.5) * 6))
target = (np.random.rand(n) < default_prob).astype(int)

df_woe = pd.DataFrame({'utilization': utilization, 'target': target})
print(f"Dataset Size: {n}, Overall Default Rate: {target.mean()*100:.2f}%")

Dataset Size: 2000, Overall Default Rate: 48.75%

## 1. Calculating WoE and Information Value (IV)

In [1]:
def calculate_woe_iv(df, feature, target_col, bins=5):
    # Discretize continuous feature into quantile bins
    df_temp = df.copy()
    df_temp['bin'] = pd.qcut(df_temp[feature], q=bins, duplicates='drop')
    
    total_goods = (df_temp[target_col] == 0).sum()
    total_bads = (df_temp[target_col] == 1).sum()
    
    grouped = df_temp.groupby('bin', observed=False).agg(
        goods=(target_col, lambda x: (x == 0).sum()),
        bads=(target_col, lambda x: (x == 1).sum())
    ).reset_index()
    
    grouped['dist_good'] = grouped['goods'] / total_goods
    grouped['dist_bad'] = grouped['bads'] / total_bads
    
    grouped['WoE'] = np.log((grouped['dist_good'] + 1e-9) / (grouped['dist_bad'] + 1e-9))
    grouped['IV_contribution'] = (grouped['dist_good'] - grouped['dist_bad']) * grouped['WoE']
    
    total_iv = grouped['IV_contribution'].sum()
    return grouped, total_iv

woe_table, iv_val = calculate_woe_iv(df_woe, 'utilization', 'target', bins=5)

print("=== WEIGHT OF EVIDENCE (WoE) TABLE ===")
display_woe = woe_table[['bin', 'goods', 'bads', 'dist_good', 'dist_bad', 'WoE', 'IV_contribution']]
print(display_woe.to_string(index=False))
print(f"\nTotal Information Value (IV): {iv_val:.4f}")
print(f"Predictive Power Assessment:  {'Strong Predictor' if 0.30 <= iv_val <= 0.50 else 'Very Strong Predictor' if iv_val > 0.50 else 'Moderate'}")

=== WEIGHT OF EVIDENCE (WoE) TABLE ===
                  bin  goods  bads  dist_good  dist_bad    WoE  IV_contribution
(0.0491, 0.231]    342    58      0.334     0.059  1.724            0.474
  (0.231, 0.407]    278   122      0.271     0.125  0.774            0.113
  (0.407, 0.589]    205   195      0.200     0.200  0.000            0.000
  (0.589, 0.772]    134   266      0.131     0.273 -0.734            0.104
  (0.772, 0.950]     66   334      0.064     0.343 -1.678            0.468

Total Information Value (IV): 1.1590
Predictive Power Assessment:  Very Strong Predictor